# 동학개미는 정말 이겼을까: 5년치 순매수 데이터의 답

## 프로젝트 목표

**핵심 질문**: "개인 투자자가 가장 많이 순매수한 종목을 따라 샀다면, 그냥 코스피 ETF를 산 것보다 나았을까?"

### 사전 가설

| 항목 | 사전 가설 | 검증 방법 |
|---|---|---|
| 개미 상위 종목 vs 코스피 | 시기 의존적일 것 | 연도별 수익률 비교 |
| 개미는 고점에서 살 것 | 가능성 높음 | 순매수 시점 vs 주가 위치 |
| 시기별 결론이 다를 것 | 그럴 것 | 2020~2025 연도별 분리 분석 |

### 이 노트북의 역할
1. pykrx로 연도별 개인 순매수 상위 종목 추출
2. 해당 종목들의 일별 주가 수집
3. 코스피 지수 수집
4. 수집 데이터 품질 점검
5. 분석용 CSV 저장

In [37]:
import pandas as pd
import numpy as np
from pykrx import stock
from datetime import datetime
from time import sleep
import warnings
import os

warnings.filterwarnings('ignore')

# 데이터 저장 경로
DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

print("라이브러리 로드 완료")
print(f"데이터 저장 경로: {os.path.abspath(DATA_DIR)}")

# 종목코드 형식 통일 (string + zero-padded)
prices['ticker'] = prices['ticker'].astype(str).str.zfill(6)
top_stocks['종목코드'] = top_stocks['종목코드'].astype(str).str.zfill(6)

# 매칭 검증
unmatched = set(top_stocks['종목코드'].unique()) - set(prices['ticker'].unique())
if unmatched:
    print(f"⚠️ prices에 없는 종목코드: {unmatched}")
else:
    print("✅ 모든 종목코드가 prices와 매칭됨")

라이브러리 로드 완료
데이터 저장 경로: c:\donghakgaemi-analysis\data
✅ 모든 종목코드가 prices와 매칭됨


## 1. 개인 순매수 상위 종목 로드

KRX에서 다운로드한 연도별 개인 투자자 순매수 상위 종목 데이터를 로드합니다.

In [28]:
# 연도별 CSV 로드
YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

all_data = []

for year in YEARS:
    filepath = f'../data/net_purchases_{year}.csv'
    df = pd.read_csv(filepath, encoding='cp949')
    df['year'] = year
    all_data.append(df)
    print(f"{year}년: {len(df)}종목 로드")

net_purchases = pd.concat(all_data, ignore_index=True)

print(f"\n전체: {len(net_purchases)}건")
print(f"컬럼: {list(net_purchases.columns)}")
net_purchases.head()

2020년: 929종목 로드
2021년: 940종목 로드
2022년: 946종목 로드
2023년: 960종목 로드
2024년: 958종목 로드
2025년: 959종목 로드

전체: 5692건
컬럼: ['종목코드', '종목명', '거래량_매도', '거래량_매수', '거래량_순매수', '거래대금_매도', '거래대금_매수', '거래대금_순매수', 'year']


,종목코드,종목명,거래량_매도,거래량_매수,거래량_순매수,거래대금_매도,거래대금_매수,거래대금_순매수,year
0,005930,삼성전자,1599948818,1777118118,177169300,90650166661550,100245348044650,9595181383100,2020
1,005935,삼성전자우,264104991,376253599,112148608,13549378702150,19650688420250,6101309718100,2020
2,005380,현대차,266039842,287136269,21096427,37402400321300,39992372048900,2589971727600,2020
3,035420,NAVER,74230248,82119282,7889034,19362448495000,21414918105000,2052469610000,2020
4,055550,신한지주,203277379,246800319,43522940,6356278836450,7649362698050,1293083861600,2020


In [29]:
# 연도별 순매수 상위 20종목 추출
TOP_N = 20

top_stocks = net_purchases.groupby('year').apply(
    lambda x: x.nlargest(TOP_N, '거래대금_순매수')
).reset_index(drop=True)

print(f"연도별 상위 {TOP_N}종목: 총 {len(top_stocks)}건")
print(f"고유 종목 수: {top_stocks['종목코드'].nunique()}개")

# 연도별 1위 확인
print(f"\n연도별 순매수 1위:")
for year in YEARS:
    row = top_stocks[top_stocks['year'] == year].iloc[0]
    print(f"  {year}년: {row['종목명']} ({row['거래대금_순매수']:,.0f}원)")

연도별 상위 20종목: 총 120건
고유 종목 수: 67개

연도별 순매수 1위:
  2020년: 삼성전자 (9,595,181,383,100원)
  2021년: 삼성전자 (31,223,867,688,000원)
  2022년: 삼성전자 (16,070,270,996,300원)
  2023년: POSCO홀딩스 (11,332,382,051,000원)
  2024년: 삼성전자 (12,092,257,507,300원)
  2025년: NAVER (3,354,637,722,600원)


## 2. 주가 데이터 수집

상위 종목들의 일별 주가(OHLCV)를 pykrx로 수집합니다.
각 종목의 연간 수익률을 계산하기 위해 해당 연도의 주가가 필요합니다.

In [30]:
# 수집 대상 종목 리스트
unique_tickers = top_stocks['종목코드'].unique()
print(f"주가 수집 대상: {len(unique_tickers)}개 종목")
print(f"수집 기간: 2020-01-01 ~ 2025-12-31")

# 종목별 주가 수집
all_prices = []

for i, ticker in enumerate(unique_tickers):
    name = top_stocks[top_stocks['종목코드'] == ticker]['종목명'].iloc[0]
    print(f"  [{i+1}/{len(unique_tickers)}] {name} ({ticker})", end=" ")
    
    try:
        df = stock.get_market_ohlcv("20200101", "20251231", ticker)
        df['ticker'] = ticker
        df['종목명'] = name
        df = df.reset_index()
        all_prices.append(df)
        print(f"→ {len(df)}일")
    except Exception as e:
        print(f"→ ❌ {e}")
    
    sleep(0.5)  # KRX 서버 부하 방지

prices = pd.concat(all_prices, ignore_index=True)
print(f"\n수집 완료: {len(prices):,}행, {prices['ticker'].nunique()}종목")

주가 수집 대상: 67개 종목
수집 기간: 2020-01-01 ~ 2025-12-31
  [1/67] 삼성전자 (005930) → 1473일
  [2/67] 삼성전자우 (005935) → 1473일
  [3/67] 현대차 (005380) → 1473일
  [4/67] NAVER (035420) → 1473일
  [5/67] 신한지주 (055550) → 1473일
  [6/67] 카카오 (035720) → 1473일
  [7/67] SK (034730) → 1473일
  [8/67] 한국전력 (015760) → 1473일
  [9/67] SK하이닉스 (000660) → 1473일
  [10/67] KT&G (033780) → 1473일
  [11/67] SK텔레콤 (017670) → 1473일
  [12/67] KB금융 (105560) → 1473일
  [13/67] 현대모비스 (012330) → 1473일
  [14/67] SK이노베이션 (096770) → 1473일
  [15/67] S-Oil (010950) → 1473일
  [16/67] 기아차 (000270) → 1473일
  [17/67] SK바이오팜 (326030) → 1349일
  [18/67] 기업은행 (024110) → 1473일
  [19/67] 호텔신라 (008770) → 1473일
  [20/67] 빅히트 (352820) → 1279일
  [21/67] LG전자 (066570) → 1473일
  [22/67] 금호석유 (011780) → 1473일
  [23/67] 엔씨소프트 (036570) → 1473일
  [24/67] 셀트리온 (068270) → 1473일
  [25/67] LG생활건강 (051900) → 1473일
  [26/67] POSCO (005490) → 1473일
  [27/67] 삼성전기 (009150) → 1473일
  [28/67] SK바이오사이언스 (302440) → 1175일
  [29/67] 두산에너빌리티 (034020) → 1473일
  [30/67] 카카오뱅크

In [31]:
# 코스피 지수 수집 (yfinance)
import yfinance as yf

print("코스피 지수 수집 중...")
kospi_raw = yf.download("^KS11", start="2020-01-01", end="2025-12-31")
kospi = kospi_raw.reset_index()
kospi.columns = ['날짜', '종가', '고가', '저가', '시가', '거래량']
kospi = kospi[['날짜', '시가', '고가', '저가', '종가', '거래량']]

print(f"코스피 지수: {len(kospi)}일")
print(f"기간: {kospi['날짜'].min()} ~ {kospi['날짜'].max()}")
kospi.head()

[*********************100%***********************]  1 of 1 completed

코스피 지수 수집 중...
코스피 지수: 1471일
기간: 2020-01-02 00:00:00 ~ 2025-12-30 00:00:00


,날짜,시가,고가,저가,종가,거래량
0,2020-01-02,2201.209961,2202.320068,2171.840088,2175.169922,494700
1,2020-01-03,2192.580078,2203.379883,2165.389893,2176.459961,631600
2,2020-01-06,2154.969971,2164.419922,2149.949951,2155.070068,592700
3,2020-01-07,2166.600098,2181.620117,2164.270020,2175.540039,568200
4,2020-01-08,2156.270020,2162.320068,2137.719971,2151.310059,913800


In [32]:
# 데이터 저장
top_stocks.to_csv('../data/top_stocks.csv', index=False, encoding='utf-8-sig')
prices.to_csv('../data/prices.csv', index=False, encoding='utf-8-sig')
kospi.to_csv('../data/kospi.csv', index=False, encoding='utf-8-sig')

print("저장 완료:")
print(f"  top_stocks.csv: {len(top_stocks)}행 (연도별 순매수 상위 종목)")
print(f"  prices.csv: {len(prices):,}행 (종목별 일별 주가)")
print(f"  kospi.csv: {len(kospi)}행 (코스피 지수)")

저장 완료:
  top_stocks.csv: 120행 (연도별 순매수 상위 종목)
  prices.csv: 89,174행 (종목별 일별 주가)
  kospi.csv: 1471행 (코스피 지수)


In [33]:
# 데이터 품질 점검
print("=" * 50)
print("데이터 품질 점검")
print("=" * 50)

# 1) 결측치
print(f"\n1) 결측치")
print(f"  top_stocks: {top_stocks.isnull().sum().sum()}건")
print(f"  prices: {prices.isnull().sum().sum()}건")
print(f"  kospi: {kospi.isnull().sum().sum()}건")

# 2) 주가 0원 또는 음수
zero_prices = prices[prices['종가'] <= 0]
print(f"\n2) 주가 0원 이하: {len(zero_prices)}건")

# 3) 종목별 데이터 기간 확인
print(f"\n3) 종목별 거래일 수:")
ticker_days = prices.groupby('종목명')['날짜'].count().sort_values()
print(f"  최소: {ticker_days.iloc[0]}일 ({ticker_days.index[0]})")
print(f"  최대: {ticker_days.iloc[-1]}일 ({ticker_days.index[-1]})")
print(f"  중앙값: {ticker_days.median():.0f}일")

# 4) 연도별 종목 중복 확인
print(f"\n4) 4년 이상 상위 20에 포함된 종목:")
ticker_year_count = top_stocks.groupby('종목명')['year'].count()
repeat_stocks = ticker_year_count[ticker_year_count >= 4].sort_values(ascending=False)
for name, count in repeat_stocks.items():
    print(f"  {name}: {count}년 등장")

데이터 품질 점검

1) 결측치
  top_stocks: 0건
  prices: 14건
  kospi: 0건

2) 주가 0원 이하: 0건

3) 종목별 거래일 수:
  최소: 26일 (삼성에피스홀딩스)
  최대: 1473일 (호텔신라)
  중앙값: 1473일

4) 4년 이상 상위 20에 포함된 종목:
  NAVER: 5년 등장
  SK하이닉스: 5년 등장
  LG생활건강: 4년 등장
  삼성전자: 4년 등장
  현대차: 4년 등장


## 4. 수집 결과 요약

### 수집 데이터

| 파일 | 내용 | 규모 |
|------|------|------|
| top_stocks.csv | 연도별 개인 순매수 상위 20종목 | 120행 (6년 × 20) |
| prices.csv | 상위 종목 일별 주가 (OHLCV) | 89,174행, 67종목 |
| kospi.csv | 코스피 지수 | 1,471일 |

### 데이터 품질
- 결측치: prices 14건 (전체 대비 무시 가능)
- 이상치: 주가 0원 이하 없음
- 삼성에피스홀딩스: 26일 (신규 상장, 분석 시 유의)

### 주요 관찰
- 삼성전자가 6년 중 4년 순매수 1위
- NAVER, SK하이닉스: 5년 연속 상위 20
- 2023년 1위가 POSCO홀딩스, 2025년 1위가 NAVER로 변화

In [36]:
# 신규 상장주가 어느 보유 연도에 영향을 미쳤는지 확인
print("=" * 60)
print("신규 상장주 백테스트 영향 점검")
print("=" * 60)

for buy_year in [2020, 2021, 2022, 2023, 2024]:
    hold_year = buy_year + 1
    portfolio = top_stocks[top_stocks['year'] == buy_year].head(10)
    
    # 보유 연도 1월 첫 거래일
    kospi_year = kospi[kospi['날짜'].dt.year == hold_year].sort_values('날짜')
    if len(kospi_year) < 2:
        continue
    first_trading_day = kospi_year['날짜'].iloc[0]
    
    print(f"\n[{hold_year}년 보유 — 코스피 첫 거래일: {first_trading_day.date()}]")
    
    for _, row in portfolio.iterrows():
        stock_data = prices[
            (prices['ticker'] == row['종목코드']) &
            (prices['날짜'].dt.year == hold_year)
        ].sort_values('날짜')
        
        if len(stock_data) < 2:
            print(f"  ⚠️  {row['종목명']}: 거래일 부족 ({len(stock_data)}일)")
            continue
        
        stock_first_day = stock_data['날짜'].iloc[0]
        gap_days = (stock_first_day - first_trading_day).days
        
        if gap_days > 5:  # 5일 넘게 차이나면 신규 상장 가능성
            print(f"  ⚠️  {row['종목명']}: 첫 거래일 {stock_first_day.date()} ({gap_days}일 차이)")

신규 상장주 백테스트 영향 점검

[2021년 보유 — 코스피 첫 거래일: 2021-01-04]

[2022년 보유 — 코스피 첫 거래일: 2022-01-04]

[2023년 보유 — 코스피 첫 거래일: 2023-01-02]

[2024년 보유 — 코스피 첫 거래일: 2024-01-02]

[2025년 보유 — 코스피 첫 거래일: 2025-01-02]
